In [58]:

# in this notebook we will implement the remaining parts of the gpt model 
# we have already implemented the multihead attention mechanism
# we will need now to implement the feed forward network the layer normalization and the residual connections
# then finally stack them all together to form the transformer block
# we will also implement the GELU function (gaussian error linear unit) which is a smooth approximation of the ReLU function
# and is used as the activation function in the feed forward network
# its advantage is that it is differentiable and has a non-zero gradient for negative inputs 
# which helps with the vanishing gradient problem
# it tackles the dying relu problem by allowing negative inputs to have a small positive output
# except at approximately -0.75 


In [61]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import torch
from torch import nn

torch.manual_seed(42)

In [62]:
# a layer normalization layer is a type of normalization layer that normalizes the inputs before passing them the next layer
# this is done by subtracting the mean and dividing by the standard deviation of the inputs
# however we also multiply by learnable scale and add a learnable shift parameter to the normalized inputs 
# which will allow the model to learn optimal scale and shift if that is what will make the model perform better
# provides some kind of flexibility to the model to learn the optimal scale and shift parameters for the inputs

class LayerNorm(nn.Module):
    def __init__(self, din):
        super().__init__()

        self.gamma = nn.Parameter(torch.ones(din))
        self.beta = nn.Parameter(torch.zeros(din))

    def forward(self, X):
        mean = X.mean(dim = -1, keepdim = True)
        var = X.var(dim = -1, keepdim = True, unbiased = False)  # this prevents the bias towards the /n or /n-1 (bessels correction)
        std = torch.sqrt(var + 1e-5)  # add a small value to prevent division by zero

        norm_x = (X - mean) / std

        return self.gamma * norm_x + self.beta

In [63]:
# next up is implementing the gelu function
# and showing the difference between it and the relu one

# the GELU(X) = O(X) * X where 0 is the cummulitive distribution function of standard normal distribution 
# however the approximation of it is
# 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return 0.5 * X * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (X + 0.044715 * torch.pow(X, 3))))


In [64]:
# finally we can start implementing the feed forward network
# a fnn is just a network that takes in the context vectors and asks each "individual questions" enhancing
# each vector individually 
# by taking in the input vector then mapping it to a higher dimensional space to capture richer patterns then compressing it back

class FeedForwardNetwork(nn.Module):
    def __init__(self, din, expanding_factor = 4):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(din, expanding_factor * din),
            GELU(),
            nn.Linear(expanding_factor * din, din)
        )

    def forward(self, X):
        return self.layers(X)




In [65]:
# we can now finally start implementing the transformer architecture
# the transformer is designed to preserve the input dimensions
# this is a crucial part of it as it only takes in the input enhances them then outputs them in the same dimensions


# it consists of firstly a layernorm to prepare the inputs for the next transformation
# we also save the inputs prelayer norm as a shortcut connection
# then we pass them to the multihead attention
# enhancing them followed by a dropout to regularize the model to prevent overfitting
# then we apply a residual connection after the dropout 
# followed by phase 2 of the transformer which is the feed forward network again we first normalize the inputs
# then pass them to fnn followed by dropout then a final res connection

from src.multihead import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, din, context_length, n_heads, expanding_factor, multihead_dropout, dropout):
        super().__init__()

        self.norm1 = LayerNorm(din)
        self.multiheadattention = MultiHeadAttention(din, din, n_heads, multihead_dropout, context_length)
        self.dropout = nn.Dropout(dropout)

        self.norm2 = LayerNorm(din)
        self.fnn = FeedForwardNetwork(din, expanding_factor)


    def forward(self, X):

        shortcut = X
        X = self.norm1(X)
        X = self.multiheadattention(X)
        X = self.dropout(X)

        X = X + shortcut

        shortcut = X

        X = self.norm2(X)
        X = self.fnn(X)
        X = self.dropout(X)
        X = X + shortcut

        return X
